In [3]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

In [4]:

# Carregue seus dados em um dataframe (supondo que o nome do dataframe seja "dados")
dados = pd.read_csv("data/trainWithFlight.csv")

In [5]:
del dados["HoraData"],dados["HoraDataDest"]
X = dados.drop(["target", "flightid"], axis=1)

In [7]:
corr_df = X.corr()

In [8]:

from sklearn.preprocessing import StandardScaler

X_pd = X
scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X_pd), columns=X_pd.columns)
threshold = 0.9

while True:
    corr_mat = X_std.corr()
    corr_mat = np.abs(corr_mat)  # take absolute value of correlations
    max_corr = corr_mat.replace(1, 0).values.max()  # replace 1s with 0s for each variable's correlation with itself
    
    if max_corr < threshold:
        # No correlation exceeds the threshold
        break

    # Get pair of variables with highest correlation
    bool_mat = corr_mat >= max_corr
    feature1 = bool_mat.any().idxmax()  # get column name of one correlated feature
    feature2 = bool_mat[feature1].idxmax()  # get correlated feature name
    
    # figure out which feature from the pair to delete
    if np.mean(corr_mat[feature1]) > np.mean(corr_mat[feature2]):
        X_std = X_std.drop(columns=[feature1])
    else:
        X_std = X_std.drop(columns=[feature2])

print(X_std.head())

   flightlevel_maxmean  distancia_mean  flightlevel_stdstd  \
0            -1.345859       -0.792986           -0.819981   
1            -2.045402       -0.916754           -1.181732   
2             1.043474        0.211623            0.437299   
3            -0.373135       -0.629171            0.739763   
4             0.315534        0.385809            0.376899   

   flightlevel_stdmean  speed_stdmean  speed_stdstd  sum(flight_through_area)  
0            -0.782611       1.055091      0.411966                 -0.507064  
1            -1.738417       0.664939      0.428112                 -0.507064  
2             1.906869       1.029026      1.053696                 -0.507064  
3            -0.213791       0.944774      1.074398                 -0.507064  
4             0.570408      -0.383084     -0.936425                 -0.507064  


In [9]:
X_std

,flightlevel_maxmean,distancia_mean,flightlevel_stdstd,flightlevel_stdmean,speed_stdmean,speed_stdstd,sum(flight_through_area)
0,-1.345859,-0.792986,-0.819981,-0.782611,1.055091,0.411966,-0.507064
1,-2.045402,-0.916754,-1.181732,-1.738417,0.664939,0.428112,-0.507064
2,1.043474,0.211623,0.437299,1.906869,1.029026,1.053696,-0.507064
3,-0.373135,-0.629171,0.739763,-0.213791,0.944774,1.074398,-0.507064
4,0.315534,0.385809,0.376899,0.570408,-0.383084,-0.936425,-0.507064
...,...,...,...,...,...,...,...
262411,-2.045402,-0.916754,-1.181732,-1.738417,0.664939,0.428112,-0.507064
262412,1.043474,0.211623,0.437299,1.906869,1.029026,1.053696,-0.507064
262413,0.996262,-0.103316,-0.417019,2.085960,2.276102,-0.059922,-0.507064
262414,0.978782,0.840806,-0.230789,1.405536,-0.089079,-1.006830,-0.507064


In [13]:
X_std.fillna(-99, inplace=True)

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_std, dados["target"], test_size=0.2, random_state=42)

# fit a linear regression model on the training set
model = LinearRegression()
model.fit(X_train, y_train)

# predict on the test set
y_pred = model.predict(X_test)

# compute the RMSE of the predictions
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'RMSE: {rmse}')


RMSE: 1005.1142263366559


In [8]:
corr_df[corr_df > 0.8]
print(corr_df.columns )

,origem,destino,linha,diaSemana,hora,esperas,mean_x,sum_x,mean_y,sum_y,...,dt_radar_lambdamax,dt_radar_lambdastd,flightlevel_meanmean,flightlevel_maxmean,distancia_mean,flightlevel_stdstd,flightlevel_stdmean,speed_stdmean,speed_stdstd,sum(flight_through_area)
origem,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
destino,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
linha,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
diaSemana,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hora,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
flightlevel_stdstd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
flightlevel_stdmean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.816473,NaN,NaN,1.0,NaN,NaN,NaN
speed_stdmean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
speed_stdstd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
